In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from donnees import charger_arcep, fichier_arcep
from cibles import ajouter_succes

ANNEES = [2022, 2023, 2024, 2025]
arcep = {a: ajouter_succes(charger_arcep(fichier_arcep(annee=a))) for a in ANNEES}

for a, df in arcep.items():
    print(a, df.shape)

2022 (666018, 24)
2023 (338358, 29)
2024 (331808, 30)
2025 (354282, 31)


In [2]:
pd.DataFrame({a: df["protocole"].value_counts() for a, df in arcep.items()})

,2022,2023,2024,2025
protocole,,,,
DOWNLOAD,67846.0,33606.0,33406.0,33384
ICMP,NaN,NaN,NaN,17328
STREAM,33923.0,16951.0,16720.0,16933
ULH,NaN,NaN,NaN,16695
UPLOAD,67846.0,33619.0,33393.0,16682
WEB,496403.0,254182.0,248289.0,253260


In [3]:
for col in ["operator", "zone", "situation", "terminal"]:
    print("=====", col)
    for a, df in arcep.items():
        print(a, sorted(df[col].dropna().unique()))

===== operator
2022 ['Bouygues', 'Free', 'Orange', 'SFR']
2023 ['Bouygues', 'FREE', 'Orange', 'SFR']
2024 ['Bouygues', 'Free', 'Orange', 'SFR']
2025 ['Bouygues', 'Free', 'Orange', 'SFR']
===== zone
2022 ['Zones denses', 'Zones intermediaires', 'Zones rurales', 'Zones touristiques']
2023 ['Zones denses', 'Zones intermediaires', 'Zones rurales', 'Zones touristiques']
2024 ['Zones denses', 'Zones intermediaires', 'Zones rurales', 'Zones touristiques']
2025 ['Zones denses', 'Zones intermediaires', 'Zones rurales', 'Zones touristiques']
===== situation
2022 ['Incar', 'Indoor', 'Outdoor']
2023 ['Incar', 'Indoor', 'Outdoor']
2024 ['Incar', 'Indoor', 'Outdoor']
2025 ['Incar', 'Indoor', 'Outdoor']
===== terminal
2022 ['SM-G990B', 'SM-G991B']
2023 ['Samsung Galaxy S22', 'Samsung Galaxy Zflip 4']
2024 ['Samsung Galaxy S22', 'Samsung Galaxy S23']
2025 ['SM-F731B', 'SM-S911B', 'SM-S921B']


In [5]:
ensembles = [set(df.columns) for df in arcep.values()]
communes = set.intersection(*ensembles)
print(len(communes), "colonnes communes :")
print(sorted(communes))

print()
for a, df in arcep.items():
    print(a, "en plus :", sorted(set(df.columns) - communes))

24 colonnes communes :
['acess_duration', 'bitrate_dl', 'date_start', 'hour_start', 'instant', 'latitude_start', 'loaded_in_less_10_secondes', 'loaded_in_less_5_secondes', 'longitude_start', 'mcc_start', 'mnc_start', 'operator', 'protocole', 'quality_correct', 'quality_perfect', 'rsrp', 'situation', 'succes', 'terminal', 'territory', 'upload_ok', 'url', 'zone', 'zone_name']

2022 en plus : []
2023 en plus : ['insee_dep', 'nom_dep', 'transfert_file_size', 'video_freez_duration', 'video_initialisation_duration']
2024 en plus : ['bitrate_ul', 'insee_com', 'operating_system', 'rscp', 'rsrq', 'transfert_file_size']
2025 en plus : ['Result', 'bitrate_ul', 'insee_com', 'rscp', 'rsrq', 'techno_start', 'transfert_file_size']


In [2]:
from donnees import harmoniser_arcep

h = harmoniser_arcep(arcep)
del arcep          # libère la mémoire : les 4 tableaux bruts ne servent plus
print(h.shape)
pd.crosstab(h["terminal"], h["annee"])

(1454163, 26)


annee,2022,2023,2024,2025
terminal,,,,
Galaxy S21,274348,0,0,0
Galaxy S21 FE,283038,0,0,0
Galaxy S22,0,168884,147693,0
Galaxy S23,0,0,150599,107132
Galaxy S24,0,0,0,116867
Galaxy Z Flip4,0,119038,0,0
Galaxy Z Flip5,0,0,0,79578


In [7]:
print((1 - h.pivot_table(index="protocole", columns="annee", values="succes", aggfunc="mean")).round(3))
print()
print(h.query("protocole == 'DOWNLOAD'").groupby("annee")["bitrate_dl"].median().round(1))

annee       2022   2023   2024   2025
protocole                            
STREAM     0.095  0.061  0.064  0.079
WEB        0.091  0.119  0.083  0.089

annee
2022     43.4
2023     75.1
2024    114.2
2025    142.9
Name: bitrate_dl, dtype: float64


In [8]:
s23 = h.query("terminal == 'Galaxy S23'")

print("Débit médian DOWNLOAD, Galaxy S23 seul :")
print(s23.query("protocole == 'DOWNLOAD'").groupby("annee")["bitrate_dl"].median().round(1))
print()
print("Débit médian DOWNLOAD, tous terminaux :")
print(h.query("protocole == 'DOWNLOAD' and annee >= 2024").groupby("annee")["bitrate_dl"].median().round(1))

Débit médian DOWNLOAD, Galaxy S23 seul :
annee
2024    125.5
2025    126.8
Name: bitrate_dl, dtype: float64

Débit médian DOWNLOAD, tous terminaux :
annee
2024    114.2
2025    142.9
Name: bitrate_dl, dtype: float64


In [3]:
SORTIE = "../data/processed/arcep_2022_2025.parquet"
h.to_parquet(SORTIE, index=False)

verif = pd.read_parquet(SORTIE)
print(verif.shape, verif.equals(h))

(1454163, 26) True
